## Análisis exploratorio de datos (EDA) para realizar la limpieza de datos


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import SparkSession
#quiero definir para leer el csv
from pyspark.sql import DataFrameReader
from pyspark.sql.functions import col, count, isnan, when, mean, stddev, min, max, desc
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType 

In [15]:
# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("ProyectoFinalOlga-Adriana-Paula") \
    .getOrCreate()

sc = spark.sparkContext

DATA_PATH = "/home/jovyan/work/data/"

In [16]:
df_behaviour = spark.read.parquet(DATA_PATH + 'behavioural_raw_parquet')
df_clients = spark.read.parquet(DATA_PATH + 'clients_raw_parquet',)

In [17]:

# Mostrar esquema y primeras filas
print("=== CLIENTES ===")
print(f"Filas: {df_clients.count()}, Columnas: {len(df_clients.columns)}")
df_clients.printSchema()
df_clients.show(5, truncate=False)

print("\n=== COMPORTAMIENTO ===")
print(f"Filas: {df_behaviour.count()}, Columnas: {len(df_behaviour.columns)}")
df_behaviour.printSchema()
df_behaviour.show(10, truncate=False)

=== CLIENTES ===
Filas: 162977, Columnas: 45
root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: integer (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: double (nullable = true)
 |-- AMOUNT_PRODUCT: double (nullable = true)
 |-- INSTALLMENT: double (nullable = true)
 |-- EDUCATION: string (nullable = true)
 |-- MARITAL_STATUS: string (nullable = true)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: double (nullable = true)
 |-- AGE_IN_YEARS: double (nullable = true)
 |-- JOB_SENIORITY: double (nullable = true)
 |-- HOME_SENIORITY: double (nullable = true)
 |-- LAST_UPDATE: double (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- CAR_AGE: double (nullable = true)
 |-- FAMILY_SIZE: double (nullable = true)
 |-- REACTIVE_SCORING: double (nullable = true)
 |-- PROACTIVE_SCORING: double (nullable = true)
 |-- BEHAVIORAL_SCORING: double (nullable = true)
 

In [18]:
from pyspark.sql.functions import col, count, when, isnan
from pyspark.sql.types import DoubleType, FloatType, StringType

# Función mejorada de análisis de calidad
def analyze_data_quality(df, df_name):
    print(f"\n=== CALIDAD DE DATOS: {df_name} ===")

    # Total de filas
    total_rows = df.count()

    # 1. Nulos por columna
    print("\n1. Valores nulos por columna:")
    null_counts_exprs = []

    for c in df.columns:
        dtype = df.schema[c].dataType
        if isinstance(dtype, (DoubleType, FloatType)):
            # Contar nulls + NaN
            expr = count(when(col(c).isNull() | isnan(c), c)).alias(c)
        else:
            expr = count(when(col(c).isNull(), c)).alias(c)
        
        null_counts_exprs.append(expr)

    null_counts_df = df.select(null_counts_exprs)
    null_counts_df.show(vertical=True)

    # 2. Porcentajes de nulos
    print("\n2. Porcentajes de nulos:")
    null_percentages = {}

    for c in df.columns:
        if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
            null_count = df.filter(col(c).isNull() | isnan(c)).count()
        else:
            null_count = df.filter(col(c).isNull()).count()

        percentage = (null_count / total_rows) * 100
        null_percentages[c] = percentage

        if percentage > 0:
            print(f"   {c}: {null_count} nulos ({percentage:.2f}%)")

    # 3. Filas duplicadas
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"\n3. Filas duplicadas: {duplicate_count}")

    # 4. Posibles problemas de tipo en columnas string
    print("\n4. Columnas string que parecen numéricas:")
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            sample = (
                df.select(col(field.name))
                  .filter(col(field.name).isNotNull())
                  .limit(20)
                  .toPandas()[field.name]
                  .astype(str)
            )

            numeric_like = sample.str.match(r'^\d+(\.\d+)?$').any()

            if numeric_like:
                print(f"   {field.name}: String pero contiene valores numéricos")

    return null_percentages

# Ejecutar análisis
null_clients = analyze_data_quality(df_clients, "CLIENTES")
null_behaviour = analyze_data_quality(df_behaviour, "COMPORTAMIENTO")



=== CALIDAD DE DATOS: CLIENTES ===

1. Valores nulos por columna:
-RECORD 0-----------------------------
 CLIENT_ID                   | 0      
 NON_COMPLIANT_CONTRACT      | 0      
 NAME_PRODUCT_TYPE           | 0      
 GENDER                      | 0      
 TOTAL_INCOME                | 0      
 AMOUNT_PRODUCT              | 0      
 INSTALLMENT                 | 7      
 EDUCATION                   | 39640  
 MARITAL_STATUS              | 2      
 HOME_SITUATION              | 0      
 REGION_SCORE                | 0      
 AGE_IN_YEARS                | 0      
 JOB_SENIORITY               | 29174  
 HOME_SENIORITY              | 0      
 LAST_UPDATE                 | 0      
 OWN_INSURANCE_CAR           | 0      
 CAR_AGE                     | 107550 
 FAMILY_SIZE                 | 2      
 REACTIVE_SCORING            | 91901  
 PROACTIVE_SCORING           | 337    
 BEHAVIORAL_SCORING          | 32246  
 DAYS_LAST_INFO_CHANGE       | 1      
 NUMBER_OF_PRODUCTS          | 21903

In [19]:
# Celda 6: Estadísticas para columnas numéricas
print("=== ESTADÍSTICAS DESCRIPTIVAS - CLIENTES ===")

# Seleccionar columnas numéricas
numeric_cols = [f.name for f in df_clients.schema.fields 
                if isinstance(f.dataType, (IntegerType, DoubleType))]

# Calcular estadísticas
for col_name in numeric_cols[:10]:  # Primeras 10 para no saturar
    stats = df_clients.select(
        mean(col(col_name)).alias("media"),
        stddev(col(col_name)).alias("std"),
        min(col(col_name)).alias("min"),
        max(col(col_name)).alias("max")
    ).collect()[0]
    
    print(f"\n{col_name}:")
    print(f"  Media: {stats['media']:.2f}")
    print(f"  Std: {stats['std']:.2f}")
    print(f"  Min: {stats['min']:.2f}")
    print(f"  Max: {stats['max']:.2f}")
    
    # Contar ceros o valores atípicos
    zero_count = df_clients.filter(col(col_name) == 0).count()
    if zero_count > 0:
        print(f"  Valores cero: {zero_count}")

=== ESTADÍSTICAS DESCRIPTIVAS - CLIENTES ===

NON_COMPLIANT_CONTRACT:
  Media: 0.08
  Std: 0.27
  Min: 0.00
  Max: 1.00
  Valores cero: 149741

TOTAL_INCOME:
  Media: 2029.34
  Std: 3722.50
  Min: 307.80
  Max: 1404000.00

AMOUNT_PRODUCT:
  Media: 7193.39
  Std: 4831.53
  Min: 540.00
  Max: 48486.19

INSTALLMENT:
  Media: 325.58
  Std: 173.48
  Min: 19.39
  Max: 2898.15

REGION_SCORE:
  Media: 0.02
  Std: 0.01
  Min: 0.00
  Max: 0.06

AGE_IN_YEARS:
  Media: 43.95
  Std: 11.93
  Min: 20.52
  Max: 69.08

JOB_SENIORITY:
  Media: 2398.92
  Std: 2360.50
  Min: 1.00
  Max: 17729.00

HOME_SENIORITY:
  Media: 4987.49
  Std: 3520.92
  Min: 0.00
  Max: 24044.00
  Valores cero: 43

LAST_UPDATE:
  Media: 2992.18
  Std: 1510.22
  Min: 0.00
  Max: 6874.00
  Valores cero: 7

CAR_AGE:
  Media: 11.99
  Std: 11.80
  Min: 0.00
  Max: 64.50
  Valores cero: 1136


In [20]:
# Celda 7: Análisis de columnas categóricas
print("=== ANÁLISIS CATEGÓRICAS - CLIENTES ===")

categorical_cols = [f.name for f in df_clients.schema.fields 
                    if isinstance(f.dataType, StringType)]

for col_name in categorical_cols[:8]:  # Primeras 8
    print(f"\n{col_name}:")
    
    # Conteo de categorías
    value_counts = df_clients.groupBy(col_name).count().orderBy(desc("count"))
    
    print(f"  Número de categorías únicas: {value_counts.count()}")
    
    # Mostrar top 5 categorías
    if value_counts.count() > 10:
        print("  (Mostrando top 10 categorías)")
    
    for row in value_counts.take(10):
        print(f"    {row[col_name]}: {row['count']} ({row['count']/df_clients.count()*100:.1f}%)")

=== ANÁLISIS CATEGÓRICAS - CLIENTES ===

CLIENT_ID:
  Número de categorías únicas: 162977
  (Mostrando top 10 categorías)
    ES182234856M: 1 (0.0%)
    ES182161992N: 1 (0.0%)
    ES182369296D: 1 (0.0%)
    ES182101769P: 1 (0.0%)
    ES182300407P: 1 (0.0%)
    ES182364023V: 1 (0.0%)
    ES182258667P: 1 (0.0%)
    ES182145473H: 1 (0.0%)
    ES182330438C: 1 (0.0%)
    ES182189909E: 1 (0.0%)

NAME_PRODUCT_TYPE:
  Número de categorías únicas: 2
    PRODUCT 1: 147470 (90.5%)
    PRODUCT 2: 15507 (9.5%)

GENDER:
  Número de categorías únicas: 2
    F: 107358 (65.9%)
    M: 55619 (34.1%)

EDUCATION:
  Número de categorías únicas: 5
    Secondary: 115824 (71.1%)
    None: 39640 (24.3%)
    Incomplete University: 5407 (3.3%)
    Primary School: 2017 (1.2%)
    Master/PhD: 89 (0.1%)

MARITAL_STATUS:
  Número de categorías únicas: 3
    Married: 120022 (73.6%)
    Single: 42953 (26.4%)
    None: 2 (0.0%)

HOME_SITUATION:
  Número de categorías únicas: 6
    House: 144579 (88.7%)
    Living with r

In [25]:
# ============================================================================
# 6. LIMPIEZA Y TRANSFORMACIÓN DE DATOS
# ============================================================================

print("\n" + "=" * 80)
print("LIMPIEZA Y TRANSFORMACIÓN DE DATOS")
print("=" * 80)

# 6.1 Limpieza de CLIENT.CSV
print("\n🧹 6.1 LIMPIANDO CLIENT.CSV...")

# 6.1.1 Eliminar duplicados manteniendo el primer registro
df_clients_clean = df_clients.dropDuplicates(["CLIENT_ID"])

# 6.1.2 Tratar valores faltantes
# Para variables categóricas: imputar con "UNKNOWN" o moda
# Para variables numéricas: imputar con mediana o 0 según contexto

# Columnas categóricas
categorical_cols = ["GENDER", "MARITAL_STATUS", "EDUCATION", 
                    "HOME_SITUATION", "OCCUPATION", 
                    "EMPLOYER_ORGANIZATION_TYPE", "NAME_PRODUCT_TYPE"]

# Imputar valores faltantes en categóricas
for col in categorical_cols:
    if col in df_clients_clean.columns:
        # Calcular moda
        mode_value = df_clients_clean.groupBy(col).count() \
            .orderBy(F.desc("count")) \
            .first()[0] if df_clients_clean.filter(F.col(col).isNotNull()).count() > 0 else "UNKNOWN"
        
        # Imputar
        df_clients_clean = df_clients_clean.fillna({col: mode_value})

# Columnas numéricas - imputar con 0 o mediana según el caso
numeric_cols = ["AGE_IN_YEARS", "FAMILY_SIZE", "JOB_SENIORITY", 
                "HOME_SENIORITY", "TOTAL_INCOME", "CAR_AGE",
                "NUMBER_OF_PRODUCTS", "INSTALLMENT"]

for col in numeric_cols:
    if col in df_clients_clean.columns:
        # Para algunas columnas, 0 tiene sentido (ej: CAR_AGE si no tiene coche)
        if col in ["CAR_AGE", "NUMBER_OF_PRODUCTS", "INSTALLMENT"]:
            df_clients_clean = df_clients_clean.fillna({col: 0})
        else:
            # Calcular mediana
            median_value = df_clients_clean.approxQuantile(col, [0.5], 0.01)[0]
            df_clients_clean = df_clients_clean.fillna({col: median_value})

# 6.1.3 Corrección de tipos de datos
# Convertir columnas booleanas
boolean_cols = ["OWN_INSURANCE_CAR", "DIGITAL_CLIENT", "HOME_OWNER", 
                "NON_COMPLIANT_CONTRACT"]

for col in boolean_cols:
    if col in df_clients_clean.columns:
        df_clients_clean = df_clients_clean.withColumn(
            col,
            F.when(
                F.col(col).cast("string").isin(["1", "True", "true"]),
                True
            ).otherwise(False)
        )

# Convertir fechas y tiempos a tipo timestamp
date_cols = ["LAST_UPDATE", "DAYS_LAST_INFO_CHANGE"]
for col in date_cols:
    if col in df_clients_clean.columns:
        df_clients_clean = df_clients_clean.withColumn(
            f"{col}_DAYS",
            F.col(col).cast("int")
        )

print("✅ Limpieza de CLIENT.CSV completada")
print(f"Registros después de limpieza: {df_clients_clean.count():,}")

# 6.2 Limpieza de BEHAVIOURAL.CSV
print("\n🧹 6.2 LIMPIANDO BEHAVIOURAL.CSV...")

# 6.2.1 Convertir DATE a tipo fecha
df_behaviour_clean = df_behaviour.withColumn(
    "DATE",
    F.to_date(F.col("DATE"), "dd/MM/yyyy")
)

# 6.2.2 Eliminar registros con fechas inválidas o futuras
from datetime import datetime
current_date = datetime.now()

df_behaviour_clean = df_behaviour_clean.filter(
    (F.col("DATE").isNotNull()) & 
    (F.col("DATE") <= F.lit(current_date))
)

# 6.2.3 Tratar valores faltantes en columnas numéricas
numeric_cols_behaviour = ["CREDIT_CARD_BALANCE", "CREDIT_CARD_LIMIT",
                         "CREDIT_CARD_DRAWINGS_ATM", "CREDIT_CARD_DRAWINGS_POS",
                         "CREDIT_CARD_DRAWINGS_OTHER", "CREDIT_CARD_DRAWINGS",
                         "CREDIT_CARD_PAYMENT", "NUMBER_DRAWINGS_ATM",
                         "NUMBER_DRAWINGS", "NUMBER_INSTALMENTS"]

for col in numeric_cols_behaviour:
    if col in df_behaviour_clean.columns:
        df_behaviour_clean = df_behaviour_clean.fillna({col: 0})

# 6.2.4 Calcular métricas derivadas
df_behaviour_clean = df_behaviour_clean.withColumn(
    "CREDIT_CARD_USAGE_RATIO",
    F.when(F.col("CREDIT_CARD_LIMIT") > 0,
           F.col("CREDICT_CARD_BALANCE") / F.col("CREDIT_CARD_LIMIT"))
     .otherwise(0)
)

df_behaviour_clean = df_behaviour_clean.withColumn(
    "AVG_DRAWING_AMOUNT",
    F.when(F.col("NUMBER_DRAWINGS") > 0,
           F.col("CREDIT_CARD_DRAWINGS") / F.col("NUMBER_DRAWINGS"))
     .otherwise(0)
)

print("✅ Limpieza de BEHAVIOURAL.CSV completada")
print(f"Registros después de limpieza: {df_behaviour_clean.count():,}")


LIMPIEZA Y TRANSFORMACIÓN DE DATOS

🧹 6.1 LIMPIANDO CLIENT.CSV...
✅ Limpieza de CLIENT.CSV completada
Registros después de limpieza: 162,977

🧹 6.2 LIMPIANDO BEHAVIOURAL.CSV...
✅ Limpieza de BEHAVIOURAL.CSV completada
Registros después de limpieza: 1,724,854


In [35]:
# ============================================================================
# 7. ANÁLISIS DE OUTLIERS Y VALORES EXTREMOS
# ============================================================================

print("\n" + "=" * 80)
print("ANÁLISIS DE OUTLIERS Y VALORES EXTREMOS")
print("=" * 80)

def analyze_outliers(df, column):
    """Analiza outliers usando el método IQR"""
    stats = df.select(
        F.mean(column).alias("mean"),
        F.stddev(column).alias("std"),
        F.min(column).alias("min"),
        F.max(column).alias("max")
    ).collect()[0]
    
    # Calcular percentiles para IQR
    quantiles = df.approxQuantile(column, [0.25, 0.5, 0.75], 0.01)
    q1, median, q3 = quantiles[0], quantiles[1], quantiles[2]
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    # Contar outliers
    outliers_lower = df.filter(F.col(column) < lower_bound).count()
    outliers_upper = df.filter(F.col(column) > upper_bound).count()
    total_outliers = outliers_lower + outliers_upper
    
    print(f"\n📊 Análisis de outliers para {column}:")
    print(f"  Media: {stats['mean']:.2f}")
    print(f"  Desviación estándar: {stats['std']:.2f}")
    print(f"  Rango: [{stats['min']}, {stats['max']}]")
    print(f"  Q1: {q1:.2f}, Mediana: {median:.2f}, Q3: {q3:.2f}")
    print(f"  IQR: {iqr:.2f}")
    print(f"  Límites: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers detectados: {total_outliers} ({total_outliers/df.count()*100:.2f}%)")
    
    return {
        "column": column,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outliers": total_outliers
    }

# Analizar outliers en columnas clave
print("\n🔍 7.1 OUTLIERS EN VARIABLES FINANCIERAS (CLIENT.CSV):")
financial_cols = ["TOTAL_INCOME", "AMOUNT_PRODUCT", "INSTALLMENT"]
outliers_info = []

for col in financial_cols:
    if col in df_clients.columns:
        outlier_info = analyze_outliers(df_clients, col)
        outliers_info.append(outlier_info)

print("\n🔍 7.2 OUTLIERS EN COMPORTAMIENTO (BEHAVIOURAL.CSV):")
behaviour_cols = ["CREDIT_CARD_BALANCE", "CREDIT_CARD_DRAWINGS", "CREDIT_CARD_PAYMENT"]

for col in behaviour_cols:
    if col in df_behaviour_clean.columns:
        analyze_outliers(df_behaviour_clean, col)


ANÁLISIS DE OUTLIERS Y VALORES EXTREMOS

🔍 7.1 OUTLIERS EN VARIABLES FINANCIERAS (CLIENT.CSV):


RuntimeError: SparkContext or SparkSession should be created first.